## 基本跑通测试

In [1]:
AGENT_SYSTEM_PROMPT = """
你是一个智能旅行助手。你的任务是分析用户的请求，并使用可用工具一步步地解决问题。

# 可用工具:
- `get_weather(city: str)`: 查询指定城市的实时天气。
- `get_attraction(city: str, weather: str)`: 根据城市和天气搜索推荐的旅游景点。

# 输出格式要求:
你的每次回复必须严格遵循以下格式，包含一对Thought和Action：

Thought: [你的思考过程和下一步计划]
Action: [你要执行的具体行动]

Action的格式必须是以下之一：
1. 调用工具：function_name(arg_name="arg_value")
2. 结束任务：Finish[最终答案]

# 重要提示:
- 每次只输出一对Thought-Action
- Action必须在同一行，不要换行
- 当收集到足够信息可以回答用户问题时，必须使用 Action: Finish[最终答案] 格式结束

请开始吧！
"""


In [ ]:
import requests
import os
from tavily import TavilyClient

# 简单的城市名到城市代码映射（可按需扩展）
CITY_CODE_MAP = {
    "北京": "101010100",
    "上海": "101020100",
    "广州": "101280101",
    "深圳": "101280601",
    "杭州": "101210101",
    "南京": "101190101",
    "天津": "101030100",
}


def get_weather(city: str) -> str:
    """使用国内免费接口 t.weather.itboy.net 查询天气信息。

    接口地址示例: http://t.weather.itboy.net/api/weather/city/101010100
    数据来源为中央气象台镜像，一般在国内网络环境下稳定可用。
    """
    city_code = CITY_CODE_MAP.get(city)
    if not city_code:
        return f"错误: 暂不支持城市 '{city}'，请在 CITY_CODE_MAP 中补充对应城市代码。"

    url = f"http://t.weather.itboy.net/api/weather/city/{city_code}"

    try:
        resp = requests.get(url, timeout=10)
        resp.raise_for_status()
        data = resp.json()

        # 状态码 200 表示成功
        if data.get("status") != 200:
            return f"错误: 查询 {city} 天气失败，接口返回: {data.get('msg', '未知错误')}"

        city_info = data.get("cityInfo", {})
        weather_data = data.get("data", {})
        forecast_list = weather_data.get("forecast", [])
        if not forecast_list:
            return f"错误: 未获取到 {city} 的预报数据。"

        today = forecast_list[0]
        city_name = city_info.get("city", city)
        weather_type = today.get("type", "未知天气")
        high = today.get("high", "")  # 形如“高温 30℃”
        low = today.get("low", "")    # 形如“低温 20℃”
        notice = today.get("notice", "")

        return f"{city_name}今天天气{weather_type}，{low}，{high}。{notice}"

    except requests.exceptions.RequestException as e:
        return f"错误: 查询 {city} 天气时遇到网络问题 - {e}"
    except Exception as e:
        return f"错误: 解析 {city} 天气数据时出现异常 - {e}"

def get_attraction(city: str, weather: str) -> str:
    """
    根据城市和天气，使用Tavily Search API搜索并返回优化后的景点推荐。
    """
    # 1. 从环境变量中读取API密钥
    api_key = os.environ.get("TAVILY_API_KEY")
    if not api_key:
        return "错误:未配置TAVILY_API_KEY环境变量。"

    # 2. 初始化Tavily客户端
    tavily = TavilyClient(api_key=api_key)
    
    # 3. 构造一个精确的查询
    query = f"'{city}' 在'{weather}'天气下最值得去的旅游景点推荐及理由"
    
    try:
        # 4. 调用API，include_answer=True会返回一个综合性的回答
        # 注意：Tavily 默认也有超时机制，这里保持默认即可
        response = tavily.search(query=query, search_depth="basic", include_answer=True)
        
        # 5. Tavily返回的结果已经非常干净，可以直接使用
        # response['answer'] 是一个基于所有搜索结果的总结性回答
        if response.get("answer"):
            return response["answer"]
        
        # 如果没有综合性回答，则格式化原始结果
        formatted_results = []
        for result in response.get("results", []):
            # 确保 result 中包含 title 和 content
            title = result.get('title', '未知标题')
            content = result.get('content', '无内容描述')
            formatted_results.append(f"- {title}: {content}")
        
        if not formatted_results:
             return "抱歉，没有找到相关的旅游景点推荐。"

        return "根据搜索，为您找到以下信息:\n" + "\n".join(formatted_results)

    except Exception as e:
        return f"错误:执行Tavily搜索时出现问题 - {e}"
        
# 将所有工具函数放入一个字典，方便后续调用
available_tools = {
    "get_weather": get_weather,
    "get_attraction": get_attraction,
}

In [16]:
import requests
import os
import re
import json
from typing import Optional, Dict, Any
from tavily import TavilyClient

# ================= 配置区域 =================

# 常用城市名称到 weather.com.cn 城市ID的映射表
# 如果用户输入的城市不在这里，代码会尝试通过搜索获取，或者提示用户。
# 数据来源：中国天气网公开接口
CITY_ID_MAP = {
    "北京": "101010100", "上海": "101020100", "广州": "101280101", "深圳": "101280601",
    "杭州": "101210101", "成都": "101270101", "重庆": "101040100", "武汉": "101200101",
    "西安": "101110101", "南京": "101190101", "天津": "101030100", "苏州": "101190501",
    "郑州": "101180101", "长沙": "101250101", "青岛": "101120201", "大连": "101070201",
    "厦门": "101230201", "昆明": "101290101", "哈尔滨": "101050101", "沈阳": "101070101",
    "济南": "101120101", "合肥": "101220101", "福州": "101220201", "南昌": "101240101",
    "贵阳": "101260101", "南宁": "101300101", "海口": "101310101", "拉萨": "101280701",
    "兰州": "101160101", "银川": "101150101", "西宁": "101270201", "乌鲁木齐": "101130101",
    "石家庄": "101090101", "太原": "101100101", "呼和浩特": "101080101", "长春": "101060101"
}

# ================= 工具函数 =================

def get_city_id(city_name: str) -> Optional[str]:
    """
    获取城市ID。
    1. 优先从本地映射表查找。
    2. 如果找不到，尝试调用中国天气网的搜索接口（部分环境可能受限）。
    """
    # 1. 查表
    if city_name in CITY_ID_MAP:
        return CITY_ID_MAP[city_name]
    
    # 2. 尝试模糊匹配（如果用户输入了"北京市"，尝试匹配"北京"）
    for name, cid in CITY_ID_MAP.items():
        if name in city_name or city_name in name:
            return cid
            
    # 3. 如果还是找不到，这里为了稳定性，不再发起额外的搜索请求（避免再次网络报错）
    # 直接返回 None，由主函数处理错误提示
    return None

def get_weather(city: str) -> str:
    """
    通过调用中国天气网 (weather.com.cn) API 查询真实的天气信息。
    解决了 wttr.in 在国内无法连接或 SSL 报错的问题。
    """
    # 1. 获取城市ID
    city_id = get_city_id(city)
    
    if not city_id:
        return f"错误:暂时无法找到城市 '{city}' 的代码。请尝试使用标准城市名（如：北京、上海、广州等）。"

    # 2. 构造实时天气API URL
    # 接口说明：http://www.weather.com.cn/data/sk/{city_id}.html
    url = f"http://www.weather.com.cn/data/sk/{city_id}.html"
    
    # 设置请求头，模拟浏览器，防止被反爬
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36",
        "Referer": "http://www.weather.com.cn/"
    }

    try:
        # 发起请求 (使用 http 协议，国内访问更稳定，无 SSL 握手问题)
        response = requests.get(url, headers=headers, timeout=10)
        response.encoding = 'utf-8'  # 确保中文不乱码
        response.raise_for_status()
        
        # 解析JSON
        # 注意：中国天气网有时会在JSON前加一些无关字符，需要清洗
        content = response.text
        # 简单的清洗逻辑：找到第一个 { 和最后一个 }
        start_idx = content.find('{')
        end_idx = content.rfind('}') + 1
        if start_idx != -1 and end_idx > start_idx:
            json_str = content[start_idx:end_idx]
            data = json.loads(json_str)
        else:
            raise ValueError("API返回的数据格式异常，无法解析JSON")

        # 提取数据
        weather_info = data.get('weatherinfo', {})
        
        if not weather_info:
            return f"错误:未获取到 {city} 的天气数据。"

        temp = weather_info.get('temp', '未知')
        weather_desc = weather_info.get('weather', '未知')
        wind_dir = weather_info.get('WD', '未知') # 风向
        wind_power = weather_info.get('WS', '未知') # 风力
        
        return f"{city}当前天气:{weather_desc}，气温{temp}摄氏度，风向{wind_dir}，风力{wind_power}"

    except requests.exceptions.RequestException as e:
        return f"错误:查询天气时遇到网络问题 - {e}"
    except (KeyError, IndexError, ValueError) as e:
        return f"错误:解析天气数据失败 - {e}"

def get_attraction(city: str, weather: str) -> str:
    """
    根据城市和天气，使用Tavily Search API搜索并返回优化后的景点推荐。
    """
    # 1. 从环境变量中读取API密钥
    api_key = os.environ.get("TAVILY_API_KEY")
    if not api_key:
        return "错误:未配置TAVILY_API_KEY环境变量。"

    # 2. 初始化Tavily客户端
    tavily = TavilyClient(api_key=api_key)
    
    # 3. 构造一个精确的查询
    query = f"'{city}' 在'{weather}'天气下最值得去的旅游景点推荐及理由"
    
    try:
        # 4. 调用API
        response = tavily.search(query=query, search_depth="basic", include_answer=True)
        
        # 5. 处理结果
        if response.get("answer"):
            return response["answer"]
        
        formatted_results = []
        for result in response.get("results", []):
            title = result.get('title', '未知标题')
            content = result.get('content', '无内容描述')
            formatted_results.append(f"- {title}: {content}")
        
        if not formatted_results:
             return "抱歉，没有找到相关的旅游景点推荐。"

        return "根据搜索，为您找到以下信息:\n" + "\n".join(formatted_results)

    except Exception as e:
        return f"错误:执行Tavily搜索时出现问题 - {e}"
        
# 将所有工具函数放入一个字典，方便后续调用
available_tools = {
    "get_weather": get_weather,
    "get_attraction": get_attraction,
}

# ================= 测试入口 (可选) =================
if __name__ == "__main__":
    # 简单测试
    test_city = "北京"
    print(f"正在查询 {test_city} 天气...")
    weather_result = get_weather(test_city)
    print(weather_result)
    
    if "错误" not in weather_result:
        print("\n正在查询景点推荐...")
        # 注意：运行此部分需要配置 TAVILY_API_KEY 环境变量
        # attraction_result = get_attraction(test_city, weather_result)
        # print(attraction_result)

正在查询 北京 天气...
北京当前天气:未知，气温18摄氏度，风向东南风，风力1级

正在查询景点推荐...


In [13]:
from openai import OpenAI

class OpenAICompatibleClient:
    """
    一个用于调用任何兼容OpenAI接口的LLM服务的客户端。
    """
    def __init__(self, model: str, api_key: str, base_url: str):
        self.model = model
        self.client = OpenAI(api_key=api_key, base_url=base_url)

    def generate(self, prompt: str, system_prompt: str) -> str:
        """调用LLM API来生成回应。"""
        print("正在调用大语言模型...")
        try:
            messages = [
                {'role': 'system', 'content': system_prompt},
                {'role': 'user', 'content': prompt}
            ]
            response = self.client.chat.completions.create(
                model=self.model,
                messages=messages,
                stream=False
            )
            answer = response.choices[0].message.content
            print("大语言模型响应成功。")
            return answer
        except Exception as e:
            print(f"调用LLM API时发生错误: {e}")
            return "错误:调用语言模型服务时出错。"


In [17]:
import re

# --- 1. 配置LLM客户端 ---
# 请根据您使用的服务，将这里替换成对应的凭证和地址
API_KEY = "sk-0c7df96744434c5d91e7a660aca149f1"
BASE_URL = "https://dashscope.aliyuncs.com/compatible-mode/v1"
MODEL_ID = "qwen3-max"
TAVILY_API_KEY="YOUR_Tavily_KEY"
os.environ['TAVILY_API_KEY'] = "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"

llm = OpenAICompatibleClient(
    model=MODEL_ID,
    api_key=API_KEY,
    base_url=BASE_URL
)

# --- 2. 初始化 ---
user_prompt = "你好，请帮我查询一下今天北京的天气，然后根据天气推荐一个合适的旅游景点。"
prompt_history = [f"用户请求: {user_prompt}"]

print(f"用户输入: {user_prompt}\n" + "="*40)

# --- 3. 运行主循环 ---
for i in range(5): # 设置最大循环次数
    print(f"--- 循环 {i+1} ---\n")
    
    # 3.1. 构建Prompt
    full_prompt = "\n".join(prompt_history)
    
    # 3.2. 调用LLM进行思考
    llm_output = llm.generate(full_prompt, system_prompt=AGENT_SYSTEM_PROMPT)
    # 模型可能会输出多余的Thought-Action，需要截断
    match = re.search(r'(Thought:.*?Action:.*?)(?=\n\s*(?:Thought:|Action:|Observation:)|\Z)', llm_output, re.DOTALL)
    if match:
        truncated = match.group(1).strip()
        if truncated != llm_output.strip():
            llm_output = truncated
            print("已截断多余的 Thought-Action 对")
    print(f"模型输出:\n{llm_output}\n")
    prompt_history.append(llm_output)
    
    # 3.3. 解析并执行行动
    action_match = re.search(r"Action: (.*)", llm_output, re.DOTALL)
    if not action_match:
        observation = "错误: 未能解析到 Action 字段。请确保你的回复严格遵循 'Thought: ... Action: ...' 的格式。"
        observation_str = f"Observation: {observation}"
        print(f"{observation_str}\n" + "="*40)
        prompt_history.append(observation_str)
        continue
    action_str = action_match.group(1).strip()

    if action_str.startswith("Finish"):
        final_answer = re.match(r"Finish\[(.*)\]", action_str).group(1)
        print(f"任务完成，最终答案: {final_answer}")
        break
    
    tool_name = re.search(r"(\w+)\(", action_str).group(1)
    args_str = re.search(r"\((.*)\)", action_str).group(1)
    kwargs = dict(re.findall(r'(\w+)="([^"]*)"', args_str))

    if tool_name in available_tools:
        observation = available_tools[tool_name](**kwargs)
    else:
        observation = f"错误:未定义的工具 '{tool_name}'"

    # 3.4. 记录观察结果
    observation_str = f"Observation: {observation}"
    print(f"{observation_str}\n" + "="*40)
    prompt_history.append(observation_str)


用户输入: 你好，请帮我查询一下今天北京的天气，然后根据天气推荐一个合适的旅游景点。
--- 循环 1 ---

正在调用大语言模型...
大语言模型响应成功。
模型输出:
Thought: 首先需要查询北京今天的天气情况，以便后续根据天气推荐合适的旅游景点。
Action: get_weather(city="北京")

Observation: 北京当前天气:未知，气温18摄氏度，风向东南风，风力1级
--- 循环 2 ---

正在调用大语言模型...
大语言模型响应成功。
模型输出:
Thought: 已获取北京的天气信息，当前天气为“未知”，但气温为18摄氏度，风力较小，适合户外活动。接下来将根据城市和天气信息推荐合适的旅游景点。
Action: get_attraction(city="北京", weather="未知")

Observation: In uncertain weather, Beijing's Forbidden City and Temple of Heaven are ideal. Both are historically significant and weatherproof.
--- 循环 3 ---

正在调用大语言模型...
大语言模型响应成功。
模型输出:
Thought: 已获取到推荐的旅游景点信息，结合天气情况，可以为用户推荐合适的景点。
Action: Finish[根据当前天气情况，推荐您游览北京的故宫和天坛。这两个景点都具有深厚的历史意义，并且适合在不确定的天气条件下参观。]

任务完成，最终答案: 根据当前天气情况，推荐您游览北京的故宫和天坛。这两个景点都具有深厚的历史意义，并且适合在不确定的天气条件下参观。
